# Production  - New order anomaly scoring

This notebook loads the trained Isolation Forest models, prepares a new order for scoring, and runs the same inference logic used in the previous analysis notebooks.

### Main goals
- load the trained anomaly detection models and supporting statistics
- validate a new order against historical combinations
- compute shipping and discount anomaly features for inference
- score the new order using the trained Isolation Forest ensemble
- classify the anomaly type and generate a human-readable explanation
- save the final inference result for downstream use

In [1]:
import numpy as np
import pandas as pd
import warnings

import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest

from pathlib import Path
from config.settings import DATA_DIR, RESULTS_FIG, RESULTS_TAB

In [2]:
warnings.filterwarnings('ignore')

data_dir = Path(DATA_DIR)
results_fig = Path(RESULTS_FIG)
results_tab = Path(RESULTS_TAB)
model_dir = results_tab / 'models'

---
# Load trained assets and reference data

In [3]:
# Load model features list
model_features = joblib.load(model_dir / 'iforest_model_features.pkl')

In [4]:
# Load scaler
scaler = joblib.load(model_dir / 'iforest_scaler.pkl')

In [5]:
# Load all Isolation Forest models
seeds = [42, 100, 500, 1000, 1500, 2000]
iforest_models = {}

for seed in seeds:
    iforest_models[seed] = joblib.load(
        model_dir / f'iforest_seed_{seed}.pkl')

In [6]:
# Load reference statistics for shipping and discount
shipping_stats = pd.read_parquet(data_dir / 'shipping_stats.parquet')
discount_stats = pd.read_parquet(data_dir / 'discount_stats.parquet')

In [7]:
# Load the anomaly_df to establish ranking
anomaly_df = pd.read_parquet(data_dir / 'anomaly_df.parquet')

# Calculate top 1% threshold
top_n = round(anomaly_df.shape[0] * .01, 0)
threshold = anomaly_df[anomaly_df['anomaly_rank'] == top_n]['iforest'].item()

print(f'All assets successfully loaded.')

All assets successfully loaded.


---
# New order

In [8]:
# Define new order with user-editable values
new_order = {
    'Days for shipping (real)': 5,
    'Days for shipment (scheduled)': 4,
    'Department Name': 'Fitness',
    'Market': 'Europe',
    'order date (DateOrders)': '2026-09-20',
    'Order Item Discount': 50.0,
    'Order Item Product Price': 159.99,
    'Order Item Quantity': 1,
    'Order Item Total': 109.99,
    'Order Status': 'COMPLETE',
    'Product Card Id': 44,
    'Product Category Id': 3,
    'Shipping Mode': 'First Class' 
}

new_df = pd.DataFrame([new_order])

In [9]:
# Define key variables for historical verification
validation_columns = ['Department Name', 'Product Card Id', 'Product Category Id']

# Validate the new order against historical data
valid_combinations = set(
    anomaly_df[validation_columns].apply(tuple, axis=1)
)
order_tuple = tuple(new_order[col] for col in validation_columns)

if order_tuple in valid_combinations:
    print('✅ Validation passed - combination exists!')
else:
    print('❌ ERROR - this combination does not exist in history!\n')

    # Check each individual field
    fields_to_check = {col: new_order[col] for col in validation_columns}
    errors = []
    
    for field_name, field_value in fields_to_check.items():
        if field_value not in anomaly_df[field_name].values:
            errors.append(field_name)
            print(f"❌ {field_name}: '{field_value}' DOES NOT EXIST")
            print(f'   Valid values: {sorted(anomaly_df[field_name].unique().tolist())}\n')
        else:
            print(f"✅ {field_name}: '{field_value}' OK")
    
    # If all individual values are valid, but the combination is not
    if not errors:
        print(f'\n⚠️  Individual values are valid, but the COMBINATION does not exist!')
        
        # Show valid combinations for the given department
        dept_name = new_order[validation_columns[0]]
        valid_in_dept = anomaly_df[anomaly_df['Department Name'] == dept_name][
            validation_columns
        ].drop_duplicates().sort_values(validation_columns[1:])
        
        print(f"Valid combinations for department '{dept_name}':")
        print(valid_in_dept.to_string(index=False))

    # ❌ STOP - the code does not end here!
    raise ValueError('❌ VALIDATION FAILED - please fix the order data')

✅ Validation passed - combination exists!


### Calculate Shipping Features

In [10]:
# Calculate shipping delay
new_df['shipping_delay'] = (
    new_df['Days for shipping (real)'] - new_df['Days for shipment (scheduled)']
)

# Join shipping stats by [Shipping Mode, Market]
new_df = new_df.merge(
    shipping_stats.reset_index(),
    on=['Shipping Mode', 'Market'],
    how='left'
)

# Shipping deviation from the group's median behavior across shipping mode and market
new_df['shipping_deviation'] = (
    new_df['shipping_delay'] - new_df['shipping_median']
)

# Robustly scaled shipping deviation; core signal for anomaly detection
new_df['shipping_anomaly_raw'] = (
    new_df['shipping_deviation'].abs() / (new_df['temp_shipping_iqr'] + 1e-5)
)

print(f"Shipping median for group: {new_df['shipping_median'].values[0]}")
print(f"Shipping deviation: {new_df['shipping_deviation'].values[0]}")
print(f"Shipping anomaly score: {new_df['shipping_anomaly_raw'].values[0]:.6f}")

Shipping median for group: 1.0
Shipping deviation: 0.0
Shipping anomaly score: 0.000000


### Calculate Discount Features

In [11]:
# Calculate discount rate
new_df['discount_rate'] = (
    new_df['Order Item Discount'] / 
    (new_df['Order Item Product Price'] * new_df['Order Item Quantity'])
)

# Join discount stats by Product Card Id
new_df = new_df.merge(
    discount_stats.reset_index(),
    on='Product Card Id',
    how='left'
)

# Discount rate deviation from the product's median historical discount
new_df['discount_deviation'] = (
    new_df['discount_rate'] - new_df['discount_median']
)

# Robustly scaled discount deviation; captures pricing anomalies at order level
new_df['discount_anomaly_raw'] = (
    new_df['discount_deviation'].abs() / (new_df['temp_discount_iqr'] + 1e-5)
)

print(f"Discount median for product: {new_df['discount_median'].values[0]:.6f}")
print(f"Discount deviation: {new_df['discount_deviation'].values[0]:.6f}")
print(f"Discount anomaly score: {new_df['discount_anomaly_raw'].values[0]:.6f}")

Discount median for product: 0.100017
Discount deviation: 0.212503
Discount anomaly score: 1.820994


### Calculate anomaly features

In [12]:
# Extract only the model features
X_new = new_df[model_features].values

# Apply RobustScaler
X_new_scaled = scaler.transform(X_new)

# Score new order with each Isolation Forest model
iforest_scores = []

for seed in seeds:
    model = iforest_models[seed]
    
    # score_samples() returns negative anomaly scores, negate for convention
    score = -model.score_samples(X_new_scaled)[0]
    iforest_scores.append(score)

# Calculate average score
iforest_avg = np.mean(iforest_scores)

In [13]:
# Check if new order is in top 1%
is_top = iforest_avg >= threshold

if is_top:
    print(f'New order has strong anomaly score = {iforest_avg:.3f}')
else:
    print('New order is ordinary with no anomaly detected')

New order has strong anomaly score = 0.721


---
# Summary

In [14]:
# Extract deviation values
shipping_dev = new_df['shipping_deviation'].item()
discount_dev = new_df['discount_deviation'].item()

# Create explanation texts
shipping_text = f'shipping {shipping_dev:.2f} days'
discount_text = f'discount {(100*discount_dev):.2f}%'

# Determine anomaly type
shipping_anomaly = new_df['shipping_anomaly_raw'].item()
discount_anomaly = new_df['discount_anomaly_raw'].item()

if max(shipping_anomaly, discount_anomaly) < 0.5:
    anomaly_type = 'none'
    deviation_explanation = 'Order does not contain anomalies'
elif abs(shipping_anomaly - discount_anomaly) < 0.1:
    anomaly_type = 'mixed'
    deviation_explanation = f'{shipping_text}. {discount_text}'
elif shipping_anomaly > discount_anomaly:
    anomaly_type = 'shipping'
    deviation_explanation = shipping_text
else:
    anomaly_type = 'discount'
    deviation_explanation = discount_text

In [15]:
# Final results summary
print('='*60)
print('ANOMALY DETECTION RESULTS FOR NEW ORDER')
print('='*60)
print(f'\nOrder Details:')
print(f'  Shipping Mode: {new_order['Shipping Mode']}')
print(f'  Market: {new_order['Market']}')
print(f'  Product Card Id: {new_order['Product Card Id']}')
print(f'  Order Status: {new_order['Order Status']}')

print(f'\nRanking:')
print(f'  In top 1%: {is_top}')

print(f'\nClassification:')
print(f'  Anomaly type: {anomaly_type}')
print(f'  Explanation: {deviation_explanation}')

ANOMALY DETECTION RESULTS FOR NEW ORDER

Order Details:
  Shipping Mode: First Class
  Market: Europe
  Product Card Id: 44
  Order Status: COMPLETE

Ranking:
  In top 1%: True

Classification:
  Anomaly type: discount
  Explanation: discount 21.25%


---
# Save results

In [16]:
# Create results dictionary
results = pd.DataFrame([{
    **new_order,
    'iforest': iforest_avg,
    'is_top_1pct': is_top,
    'anomaly_type': anomaly_type,
    'deviation_explanation': deviation_explanation
}])

results.to_csv(results_tab / 'new_order.csv', index=False)
results

,Days for shipping (real),Days for shipment (scheduled),Department Name,Market,order date (DateOrders),Order Item Discount,Order Item Product Price,Order Item Quantity,Order Item Total,Order Status,Product Card Id,Product Category Id,Shipping Mode,iforest,is_top_1pct,anomaly_type,deviation_explanation
0,5,4,Fitness,Europe,2026-09-20,50.0,159.99,1,109.99,COMPLETE,44,3,First Class,0.720995,True,discount,discount 21.25%


## Key Takeaway

- The saved Isolation Forest ensemble can score a new order consistently using the same feature pipeline as the training notebook
- Shipping and discount deviations are computed with the same group-level reference statistics, which keeps inference interpretable
- The final output is not just a score, but also a business-facing anomaly label and explanation
- The validation step is important because it protects inference from unrealistic or unseen input combinations
- The result table is ready for downstream review, reporting, or deployment